# 이미지 분류 1일차 마지막 실습 — 오늘의 생각 순서를 직접 써 보기

흔들어 본다 → 증강을 고른다 → 가려서 확인한다 → 한 조건만 바꿔 같은 시험으로 잰다 → 세 줄로 적는다

- 이 노트북 · `practice_data.npz` · `d1_base_best.pt` 세 파일이 같은 폴더에 있어야 함 (Colab: 왼쪽 파일 창에 세 파일을 올림)
- 모델은 수업에서 쓴 저장 모델(증강 없음) · 사진은 검증 사진 앞 2000장 — 장수가 달라 수업 화면 숫자와 조금 다름
- 문제마다 순서: **예상을 먼저 말로 정함 → ◆ 줄 한두 개 고침 → 실행 → 확인 질문**
- ◆ 표시가 있는 줄이 고쳐 볼 줄 · 고치지 않아도 기본값으로 끝까지 돌아감
- 막히면 각 문제 아래 "막혔을 때 이어갈 코드" 칸을 실행하고 다음 문제로 넘어감
- 난이도 — **★ 기본**(모두 · 한두 줄 고치기) · **★★ 도전**(조건 하나를 스스로 바꿔 비교) · **★★★ 심화**(짧은 코드 몇 줄 작성)
  - 수업 시간은 ★ 기준 · ★ 를 먼저 끝낸 사람만 ★★ → ★★★ 로 이어감 · 못 해도 해설을 따라가는 데 지장 없음

## 0. 준비 — 한 번만 실행

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2.functional as TF
import matplotlib.pyplot as plt

data = np.load("practice_data.npz")
x_val, y_val = torch.from_numpy(data["x_val"]), torch.from_numpy(data["y_val"])
CLASSES = [str(c) for c in data["classes"]]


class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        layers, c = [], 3
        for w in (32, 64, 128):
            layers += [nn.Conv2d(c, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(),
                       nn.Conv2d(w, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(), nn.MaxPool2d(2)]
            c = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(c, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def to_input(x):
    return (x.float() / 255 - 0.5) / 0.25


@torch.no_grad()
def predict(model, x, bs=500):
    model.eval()
    return torch.cat([model(to_input(x[k:k + bs])).softmax(1) for k in range(0, len(x), bs)])


def show(images, titles, cols=8, size=1.6):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * (size + 0.4)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, im, t in zip(axes.flat, images, titles):
        ax.imshow((im.float().clamp(0, 255) / 255).permute(1, 2, 0).numpy(), interpolation="nearest")
        ax.set_title(t, fontsize=8)
    fig.tight_layout()
    plt.show()


torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
model = SmallCNN(len(CLASSES))
model.load_state_dict(torch.load("d1_base_best.pt", map_location="cpu"))
pred = predict(model, x_val).argmax(1)
correct = (pred == y_val).nonzero().flatten()
print(f"준비 끝 · 검증 사진 {len(x_val)}장 · 모델이 맞힌 사진 {len(correct)}장")

**에러가 나면 첫 줄부터 읽을 것**
- `FileNotFoundError` — 세 파일이 노트북과 같은 폴더에 없음 · Colab 파일 창에 올렸는지 확인
- `NameError: name 'model' is not defined` — 이 준비 칸을 건너뜀 · 맨 위부터 다시 실행

## 개념 문제 — 객관식 8개 (약 5분)

- 오늘 하루 전체에서 나옴 · 보기 네 개 중 하나 · 코드를 고치기 전에 생각을 먼저 정리하는 칸
- 아래 칸을 두 번 눌러 문제를 읽고, 그 아래 코드 칸의 `answers` 에 번호(1~4)를 적은 뒤 실행 · 채점은 하지 않음 — 해설 시간에 함께 맞춰 봄

**Q1.** 증강(뒤집기 · 밝기 바꾸기 등)을 **학습 사진에만** 넣고 검증 · 시험 사진에는 넣지 않는 이유로 가장 알맞은 것은?

1) 시험에도 넣으면 정확도가 늘 올라가기 때문
2) 연습에는 변화를 주고, 비교는 매번 같은 사진 · 같은 조건으로 해야 두 모델을 견줄 수 있기 때문
3) 증강은 학습 속도를 빠르게 하는 방법이기 때문
4) 시험 사진에는 변환 함수를 쓸 수 없기 때문

**Q2.** 증강이 도움이 되는지 알고 싶다. 가장 알맞은 비교 설계는?

1) 증강을 넣으면서 에폭도 두 배로 늘려, 증강 없는 모델과 견준다
2) 증강 모델은 테스트 사진으로, 증강 없는 모델은 검증 사진으로 잰다
3) 증강 유무만 바꾸고 사진 · 모델 모양 · 시작값 · 에폭 · 시험 사진은 모두 같게 둔다
4) 두 모델을 여러 번 돌려 가장 잘 나온 한 번끼리 견준다

**Q3.** 시작값 0 으로 한 번 돌렸더니 증강 쪽이 조금 높았다. 그런데 증강 없는 모델끼리 **시작값만 바꿔도** 그보다 더 크게 움직였다. 가장 알맞은 해석은?

1) 증강이 효과가 있다고 확정한다
2) 증강이 해롭다고 결론 낸다
3) 시작값은 결과와 상관없으니 무시한다
4) 차이가 시작값 흔들림보다 작으므로 이 실험으로는 결론을 미루고, 시작값을 늘려 다시 잰다

**Q4.** 가로가 확신, 세로가 실제 정확도인 그래프에서 점들이 점선 대각선보다 **아래**에 있다. 뜻하는 것은?

1) 모델이 확신보다 더 잘 맞힘 (겸손함)
2) 모델이 실제보다 더 확신함 (과신)
3) 정확도가 낮다는 뜻일 뿐 확신과는 관계없음
4) 대각선 아래일수록 좋은 모델

**Q5.** 가려 보기에서 대상 위가 밝게 나왔다. 이것으로 "모델은 대상을 보고 판단한다"를 **증명했다고 할 수 없는** 이유로 가장 알맞은 것은?

1) 사진 몇 장 · 방법 하나이고 회색 네모 자체가 낯선 입력일 수 있어, 후보를 지지하거나 약하게 할 뿐이다
2) 가려 보기는 모델 내부를 직접 읽는 방법이라 증명이 맞다
3) 밝은 곳이 곧 모델이 보는 곳이므로 증명이 맞다
4) 가려 보기는 흑백 사진에서만 쓸 수 있다

**Q6.** 고양이 · 개 헷갈림이 해상도 때문인지 알고 싶다. 질문에 가장 맞는 잣대는?

1) 해상도를 낮춰 가며 검증 사진 전체의 정확도를 잰다
2) 해상도를 낮춰 가며 전체 사진에서 '고양이 → 개'로 틀린 장수만 센다
3) 해상도가 높을수록 좋은 것은 당연하므로 잴 필요가 없다
4) 고양이 · 개 사진만 모아 둘 중 하나로 고르게 하고, 해상도별로 얼마나 틀리는지 센다

**Q7.** 채널 수가 같을 때, 3x3 합성곱을 **두 번** 쌓은 것과 5x5 합성곱 **한 번**을 견준 설명으로 맞는 것은?

1) 한 칸이 보는 넓이는 둘 다 5x5 이고, 3x3 두 번 쪽이 학습할 숫자가 더 적고 ReLU 도 한 번 더 들어간다
2) 3x3 을 두 번 쌓아도 한 칸이 보는 넓이는 3x3 그대로다
3) 3x3 두 번 쪽이 학습할 숫자가 더 많다
4) 5x5 한 번이 언제나 더 정확하다

**Q8.** 층을 아주 깊게 쌓았더니 앞쪽(입력 가까운) 층이 거의 배우지 않는다. 가장 알맞은 설명은?

1) 앞쪽 층은 이미 다 배웠기 때문이다
2) 기울기가 뒤에서 앞으로 층마다 곱해지며 전달되는데, 1 보다 작은 값이 계속 곱해지면 앞쪽에 올수록 작아지기 때문이다
3) 사진이 작아서 앞쪽 층에 들어갈 정보가 없기 때문이다
4) 학습률을 층마다 다르게 정했기 때문이다

In [ ]:
answers = {"Q1": "", "Q2": "", "Q3": "", "Q4": "", "Q5": "", "Q6": "", "Q7": "", "Q8": ""}   # ◆ 번호(1~4)를 적을 것
print("적은 답 —", " · ".join(f"{k} {v or '-'}" for k, v in answers.items()))

## 문제 1 ★ — 변화 하나의 세기를 바꿔 보기 (흔들어 본다)

- 왜 — 수업의 흔들기 격자는 변화마다 **세기 하나**만 봄 · 세기를 바꾸면 "약하게는 괜찮고 세게만 약한가"를 가를 수 있음
- 목표 한 줄 — 회전이 아닌 변화 하나를 골라, 세기를 키울수록 답이 바뀐 비율이 어떻게 늘어나는지 그림 한 장으로 봄
- 먼저 예상 — 가장 약한 세기에서 답이 바뀐 비율은 거의 0 일까, 아닐까
- ◆ 고칠 줄 두 개 — `change` 안의 return 줄 · `strengths` 목록 (**같은 변화**를 가리키게)
- 문법 예시
  - 밝기: `(x * 배율).clamp(0, 255)` — 배율 1.0 이 원본 · 0.5 는 절반 밝기
  - 옮기기: `TF.affine(x, angle=0.0, translate=[칸, 칸], scale=1.0, shear=[0.0, 0.0])` — 칸 0 이 원본
  - 잘라 키우기: `TF.resize(TF.center_crop(x, [크기, 크기]), [32, 32], antialias=True)` — 크기 32 가 원본
- 힌트: 첫 세기는 **원본과 같은 값**으로 둘 것 (밝기 1.0 · 옮기기 0 · 잘라 키우기 32) — 이 값에서 0 이 나오면 코드가 맞게 짜인 것

In [ ]:
def change(x, strength):
    x = x.float()
    return TF.rotate(x, float(strength))            # ◆ 고른 변화로 바꿀 것 — 예: return (x * strength).clamp(0, 255)


strengths = [0, 5, 10, 20, 30, 45]                   # ◆ 고른 변화의 세기 목록 — 예: 밝기라면 [1.0, 0.8, 0.6, 0.4, 0.2]

In [ ]:
rates = []
for st in strengths:
    new = predict(model, change(x_val[correct], st)).argmax(1)
    rates.append((new != pred[correct]).float().mean().item() * 100)

plt.figure(figsize=(6, 3))
plt.plot([str(s) for s in strengths], rates, "o-")
plt.xlabel("strength")
plt.ylabel("answer changed (%)")
plt.tight_layout()
plt.show()
for st, r in zip(strengths, rates):
    print(f"세기 {st} → 답이 바뀐 비율 {r:.1f}%")

- 확인 질문 1 — 원본과 같은 세기에서 답이 바뀐 비율은 몇인가 · 왜 그래야 하는가
- 확인 질문 2 — 어느 두 세기 사이에서 가장 가파르게 늘어나는가 · 그 세기의 사진을 보면 사람 눈에도 단서가 사라졌을까
- 분모: 모델이 원본을 맞힌 사진
- 흔한 에러
  - `TypeError: rotate(): argument 'angle' ...` — return 줄은 그대로 두고 `strengths` 만 밝기 배율로 바꾼 경우와 반대 경우 모두 조심
  - 그림이 평평하게 0 — `change` 안에서 `strength` 를 쓰지 않았음

**막혔을 때 이어갈 코드** — 밝기로 바꾼 예

In [ ]:
def change(x, strength):
    return (x.float() * strength).clamp(0, 255)


strengths = [1.0, 0.8, 0.6, 0.4, 0.2]
for st in strengths:
    new = predict(model, change(x_val[correct], st)).argmax(1)
    print(f"밝기 배율 {st} → 답이 바뀐 비율 {(new != pred[correct]).float().mean().item() * 100:.1f}%")

### 문제 1 ★★ 도전 — 두 변화를 한 그림에서 견주기

- 할 일 — 문제 1 에서 고른 변화와 **다른 변화 하나**를 `change2` 로 만들어, 두 곡선을 한 그림에 그림
- 조건 하나 — 변화의 종류만 다름 · 사진 · 모델 · 분모는 그대로
- 확인 질문 — 어느 변화에 모델이 더 약한가 · 세기의 단위가 다른 두 변화를 "세기 3번째"끼리 견주는 것은 공정한가

In [ ]:
def change2(x, strength):
    return TF.affine(x.float(), angle=0.0, translate=[int(strength), int(strength)], scale=1.0, shear=[0.0, 0.0])  # ◆ 둘째 변화 — 예: 옮기기


strengths2 = [0, 1, 2, 3, 4, 6]                      # ◆ 둘째 변화의 세기 — 첫 값은 원본과 같게

rates2 = [(predict(model, change2(x_val[correct], st)).argmax(1) != pred[correct]).float().mean().item() * 100 for st in strengths2]
plt.figure(figsize=(6, 3))
plt.plot(range(len(rates)), rates, "o-", label="change 1")
plt.plot(range(len(strengths2)), rates2, "s-", label="change 2")
plt.xlabel("strength step (0 = original)")
plt.ylabel("answer changed (%)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
for st, r in zip(strengths2, rates2):
    print(f"둘째 변화 세기 {st} → 답이 바뀐 비율 {r:.1f}%")

### 문제 1 ★★★ 심화 — 가장 센 세기에서 답이 바뀐 사진 여덟 장과 새 답

- 할 일 — 아래 빈 줄 두 개를 채워, 가장 센 세기에서 답이 바뀐 사진 앞 8장을 **원본 위 · 바뀐 사진 아래**로 보이고 새 답을 제목에 적음
- 힌트 — 바뀐 사진 번호: `(new != pred[correct]).nonzero().flatten()[:8]` · 사진 번호는 `correct[j]`

In [ ]:
st = strengths[-1]
new = predict(model, change(x_val[correct], st)).argmax(1)
moved = (new != pred[correct]).nonzero().flatten()[:8]   # ◆ (채움) 답이 바뀐 사진의 자리 앞 8개
if len(moved) == 0:
    print("가장 센 세기에서도 답이 바뀐 사진이 없음")
else:
    show([x_val[correct[j]] for j in moved] + [change(x_val[correct[j]][None], st)[0] for j in moved],
         [f"true {CLASSES[y_val[correct[j]]]}" for j in moved] + [f"-> {CLASSES[new[j]]}" for j in moved], cols=len(moved))  # ◆ (채움) 제목에 새 답
    print(f"세기 {st} 에서 답이 바뀐 {int((new != pred[correct]).sum())}장 중 앞 {len(moved)}장")

## 문제 2 ★ — 내가 쓸 증강을 직접 만들어 보기

- 왜 — 수업 과제 "허용할 변환 하나와 강도"를 코드로 옮김 · 이 함수는 문제 4 의 학습에서 그대로 씀
- 목표 한 줄 — 넣을 때마다 무작위로 달라지고, **여덟 장 모두 정답 단서가 남는** 증강 함수 하나
- 먼저 예상 — 내 증강을 넣은 사진을 지금 모델(증강 없이 학습)에 보여 주면 답이 몇 장쯤 바뀔까
- ◆ 고칠 줄 — `my_augment` 안의 ◆ 두 줄 · `kind`
- 문법 예시: 좌우 반전 `x.flip(-1)` · 회전 `TF.rotate(x[None], 각도)[0]` · 무작위 수 `torch.rand(1).item()` (0~1 사이) · -a ~ +a 사이 수 `(torch.rand(1).item() * 2 - 1) * a`
- 힌트: 증강은 넣을 때마다 **무작위로** 달라져야 함 — 각도나 배율을 `torch.rand` 로 정할 것 · 고정 값(예: 늘 15도)은 증강이 아니라 한 가지 변화

In [ ]:
def my_augment(img):
    """사진 한 장(3, 32, 32)에 내가 고른 증강을 무작위로 한 번 넣음."""
    x = img.float()
    if torch.rand(1).item() < 0.5:                    # ◆ 내 변환 1 — 예: 반반 확률로 좌우 반전
        x = x.flip(-1)
    angle = (torch.rand(1).item() * 2 - 1) * 10       # ◆ 내 변환 2와 강도 — 예: -10 ~ +10도 회전
    x = TF.rotate(x[None], angle)[0]
    return x.clamp(0, 255)


kind = "horse"                                        # ◆ 보고 싶은 종류 — airplane · automobile · bird · cat · deer · dog · frog · horse · ship · truck

In [ ]:
i = int((y_val == CLASSES.index(kind)).nonzero()[0])
torch.manual_seed(0)
show([x_val[i]] + [my_augment(x_val[i]) for _ in range(7)], ["original"] + ["my augment"] * 7)

# 숫자로 한 번 더 — 맞힌 사진 300장에 내 증강을 한 번씩 넣었을 때 지금 모델의 답이 바뀐 비율
torch.manual_seed(1)
sample = correct[:300]
aug_imgs = torch.stack([my_augment(x_val[j]) for j in sample])
moved = (predict(model, aug_imgs).argmax(1) != pred[sample]).float().mean().item() * 100
print(f"내 증강을 넣은 맞힌 사진 300장 — 지금 모델의 답이 바뀐 비율 {moved:.1f}%")

- 확인 질문 1 — 여덟 장이 모두 서로 다른가 (무작위인가) · 모두 사람 눈에 같은 종류로 보이는가
- 확인 질문 2 — 답이 바뀐 비율은 "증강이 나쁘다"는 뜻인가, "모델이 아직 이 변화를 연습하지 않았다"는 뜻인가
- 흔한 에러
  - `ValueError: 'hores' is not in list` — 종류 이름 철자가 CLASSES 목록과 다름
  - `TypeError` 또는 모양 에러 — `TF.rotate` 에 넣을 때 `x[None]` 으로 묶고 `[0]` 으로 푸는 부분을 지움

**막혔을 때 이어갈 코드** — 좌우 반전만 쓰는 증강 (문제 4 에서도 이 함수를 씀)

In [ ]:
def my_augment(img):
    x = img.float()
    if torch.rand(1).item() < 0.5:
        x = x.flip(-1)
    return x.clamp(0, 255)
print("my_augment = 좌우 반전(반반) 으로 정함")

### 문제 2 ★★ 도전 — 같은 증강의 약한 판 · 센 판 견주기

- 할 일 — 내 증강의 강도 숫자 하나만 바꾼 `weak` · `strong` 두 판을 만들어, 지금 모델의 답이 바뀐 비율과 사진을 견줌
- 조건 하나 — 강도만 다름 · 같은 사진 300장 · 같은 무작위 시작값
- 확인 질문 — 강도를 키우면 비율이 느는 것은 당연함 · 그러면 강도를 고르는 기준은 비율인가, 사진 속 단서인가

In [ ]:
def aug_with(img, level):
    """좌우 반전(반반) + 밝기 (1 - level) ~ (1 + level) 배. level 이 강도."""
    x = img.float()
    if torch.rand(1).item() < 0.5:
        x = x.flip(-1)
    return (x * (1 - level + 2 * level * torch.rand(1).item())).clamp(0, 255)


weak, strong = 0.2, 0.7                               # ◆ 약한 판 · 센 판의 강도

for name, lv in (("weak", weak), ("strong", strong)):
    torch.manual_seed(1)
    imgs = torch.stack([aug_with(x_val[j], lv) for j in correct[:300]])
    r = (predict(model, imgs).argmax(1) != pred[correct[:300]]).float().mean().item() * 100
    print(f"{name} (강도 {lv}) — 지금 모델의 답이 바뀐 비율 {r:.1f}%")
torch.manual_seed(0)
show([aug_with(x_val[correct[0]], weak) for _ in range(4)] + [aug_with(x_val[correct[0]], strong) for _ in range(4)],
     ["weak"] * 4 + ["strong"] * 4)

### 문제 2 ★★★ 심화 — 새 변환 하나를 직접 짜서 넣기

- 할 일 — `my_augment2` 안에 **무작위로 조금 옮기기**(가장자리를 비춰 늘린 뒤 32칸 잘라 내기)를 몇 줄로 짬 · 수업의 학습 증강에 들어 있던 변환
- 힌트 — 늘리기 `F.pad(x[None], (4, 4, 4, 4), mode="reflect")[0]` → 크기 (3, 40, 40) · 시작 칸 `i = torch.randint(0, 9, (1,)).item()` · 잘라 내기 `x[:, i:i + 32, j:j + 32]`

In [ ]:
def my_augment2(img):
    x = img.float()
    # ◆ (채움) 가장자리를 4칸씩 비춰 늘리고, 무작위 자리에서 32칸 잘라 내기 — 지금은 비워 둠(원본 그대로)
    return x.clamp(0, 255)


torch.manual_seed(0)
show([x_val[correct[0]]] + [my_augment2(x_val[correct[0]]) for _ in range(7)], ["original"] + ["augment2"] * 7)

## 문제 3 ★ — 가려서 확인하기 (원인 후보 확인)

- 왜 — 수업(원인 후보를 실제로 확인하기)에서는 8칸 네모 하나로만 가려 봄 · 도구의 설정을 바꿔도 같은 결론이 나오는지 확인
- 목표 한 줄 — 네모 크기를 바꿔도 "밝은 곳 = 대상 위"가 유지되는지 봄
- 먼저 예상 — 네모를 4칸으로 줄이면 그림이 더 자세해질까, 더 흐릿해질까
- ◆ 고칠 줄 — `patch` (4 · 8 · 12) · `kinds` (보고 싶은 종류 네 개)
- 힌트: patch 를 하나씩 바꿔 세 번 실행하고 그림을 비교할 것

In [ ]:
def occlusion(img, label, patch=8, stride=4):
    """회색 네모를 stride 칸씩 옮기며 가리고, 정답 확률이 얼마나 떨어지는지 격자로 돌려줌."""
    base = predict(model, img[None])[0, label].item()
    n_ = (32 - patch) // stride + 1
    batch = []
    for r in range(n_):
        for c in range(n_):
            x = img.float().clone()
            x[:, r * stride:r * stride + patch, c * stride:c * stride + patch] = 128.0
            batch.append(x)
    return (base - predict(model, torch.stack(batch))[:, label]).view(n_, n_)


conf = predict(model, x_val).max(1).values


def draw(kinds, patches):
    fig, ax = plt.subplots(1 + len(patches), len(kinds), figsize=(2.3 * len(kinds), 2.4 * (1 + len(patches))), squeeze=False)
    for k, name in enumerate(kinds):
        i = int(((y_val == CLASSES.index(name)) & (pred == y_val) & (conf > 0.9)).nonzero()[0])
        img = (x_val[i].float() / 255).permute(1, 2, 0).numpy()
        ax[0, k].imshow(img, interpolation="nearest")
        ax[0, k].set_title(name, fontsize=9)
        for r, p in enumerate(patches, start=1):
            heat = occlusion(x_val[i], int(y_val[i]), patch=p)
            heat_up = F.interpolate(heat[None, None], size=(32, 32), mode="bilinear", align_corners=False)[0, 0].numpy()
            ax[r, k].imshow(img, interpolation="nearest")
            ax[r, k].imshow(heat_up.clip(0), cmap="magma", alpha=0.6)
            ax[r, k].set_title(f"patch {p}", fontsize=9)
        for a in ax[:, k]:
            a.axis("off")
    fig.tight_layout()
    plt.show()

In [ ]:
patch = 8                                             # ◆ 가리는 네모 크기 — 4 · 8 · 12
kinds = ["airplane", "ship", "horse", "cat"]          # ◆ 보고 싶은 종류 네 개

draw(kinds, [patch])

- 확인 질문 1 — 밝은 곳(가리면 정답 확률이 크게 떨어지는 곳)이 대상 위인가 배경 위인가
- 확인 질문 2 — 네모 크기를 바꿔도 밝은 곳의 자리가 같은가 · 다르다면 결론을 어떻게 적어야 하는가
- 주의: 네 장뿐 · 가리는 방법 하나 · 회색 네모 자체가 낯선 입력일 수 있음 → 원인을 증명하는 것이 아니라 후보를 지지하거나 약하게 할 뿐
- 흔한 에러
  - `IndexError: index 0 is out of bounds` — 고른 종류에서 확신 0.9 넘게 맞힌 사진이 없음 · 다른 종류로 바꿀 것
  - `ValueError: ... is not in list` — 종류 이름 철자

**막혔을 때 이어갈 코드** — 기본 설정 그대로 한 번

In [ ]:
draw(["airplane", "ship", "horse", "cat"], [8])

### 문제 3 ★★ 도전 — 가리는 색을 바꿔도 같은가

- 할 일 — 네모 크기는 8 로 두고, 가리는 색만 회색(128) → 검정(0)으로 바꿔 밝은 곳의 자리를 견줌
- 조건 하나 — 가리는 색 · 사진 · 네모 크기 · 옮기는 칸은 그대로
- 확인 질문 — 색만 바꿨는데 밝은 곳이 달라진다면, 그것은 모델에 대해 무엇을 말하는가

In [ ]:
def occlusion_fill(img, label, fill, patch=8, stride=4):
    base = predict(model, img[None])[0, label].item()
    n_ = (32 - patch) // stride + 1
    batch = []
    for r in range(n_):
        for c in range(n_):
            x = img.float().clone()
            x[:, r * stride:r * stride + patch, c * stride:c * stride + patch] = fill
            batch.append(x)
    return (base - predict(model, torch.stack(batch))[:, label]).view(n_, n_)


fills = [128.0, 0.0]                                  # ◆ 견줄 두 색 — 회색 · 검정 (255 는 흰색)
names = ["airplane", "ship", "horse", "cat"]
fig, ax = plt.subplots(len(fills), len(names), figsize=(2.3 * len(names), 2.4 * len(fills)), squeeze=False)
for k, name in enumerate(names):
    i = int(((y_val == CLASSES.index(name)) & (pred == y_val) & (conf > 0.9)).nonzero()[0])
    img = (x_val[i].float() / 255).permute(1, 2, 0).numpy()
    for r, fv in enumerate(fills):
        h = occlusion_fill(x_val[i], int(y_val[i]), fv)
        hu = F.interpolate(h[None, None], size=(32, 32), mode="bilinear", align_corners=False)[0, 0].numpy()
        ax[r, k].imshow(img, interpolation="nearest")
        ax[r, k].imshow(hu.clip(0), cmap="magma", alpha=0.6)
        ax[r, k].set_title(f"{name} · fill {int(fv)}", fontsize=8)
        ax[r, k].axis("off")
fig.tight_layout()
plt.show()

### 문제 3 ★★★ 심화 — 틀린 사진을 가려 보기 (모델의 답 쪽으로)

- 할 일 — 모델이 **틀린** 사진 한 장을 골라, 정답이 아니라 **모델의 답** 확률이 어디를 가릴 때 떨어지는지 봄 · 빈 줄 한 줄을 채움
- 힌트 — `occlusion(사진, 종류 번호)` 의 둘째 값에 정답 `y_val[i]` 대신 모델의 답 `pred[i]` 를 넣음

In [ ]:
wrong = (pred != y_val).nonzero().flatten()
i = int(wrong[conf[wrong].argmax()])                  # 가장 자신 있게 틀린 사진
heat = occlusion(x_val[i], int(y_val[i]))             # ◆ (채움) 정답 대신 모델의 답 pred[i] 로 바꿀 것
hu = F.interpolate(heat[None, None], size=(32, 32), mode="bilinear", align_corners=False)[0, 0].numpy()
fig, ax = plt.subplots(1, 2, figsize=(5, 2.7))
img = (x_val[i].float() / 255).permute(1, 2, 0).numpy()
ax[0].imshow(img, interpolation="nearest"); ax[0].set_title(f"true {CLASSES[y_val[i]]} / pred {CLASSES[pred[i]]}", fontsize=8)
ax[1].imshow(img, interpolation="nearest"); ax[1].imshow(hu.clip(0), cmap="magma", alpha=0.6); ax[1].set_title("occlusion", fontsize=8)
for a in ax:
    a.axis("off")
fig.tight_layout()
plt.show()

## 문제 4 ★ — 작은 통제 비교 직접 돌리기 (한 조건만 바꿔 같은 시험으로)

- 왜 — 수업(통제 실험)에서는 저장한 두 모델과 시작값 셋의 짧은 학습을 **보기만** 함 · 이번에는 같은 절차를 직접 한 번 돌림
- 목표 한 줄 — 문제 2 의 `my_augment` 를 **넣은 학습**과 **안 넣은 학습**을 조건 하나만 다르게 돌려, 같은 시험 사진으로 잼
- 연습용 나눔 — 이 노트북에는 학습 사진이 없으므로 **검증 사진 앞 500장을 학습용, 뒤 1000장을 시험용**으로 씀 (둘은 겹치지 않음 · 수업 모델의 성적과는 다른 연습 숫자)
- 먼저 예상 — 500장 · 8에폭처럼 짧은 학습에서 증강을 넣으면 시험 정확도가 오를까 · 차이가 크게 날까
- ◆ 고칠 줄 — `seed` 하나 · 처음에는 0 그대로 실행 → 결과를 본 뒤 1 로 바꿔 한 번 더
- 한 번 실행에 CPU 로 약 1분 · 기다리는 동안 아래 "그대로 둔 것" 목록을 읽을 것

In [ ]:
x_tr, y_tr = x_val[:500], y_val[:500]                 # 학습용 500장
x_te, y_te = x_val[1000:], y_val[1000:]               # 시험용 1000장 (겹치지 않음)


def train_small(use_aug, seed, epochs=8, bs=64):
    """처음부터 짧게 학습 → (시험용 1000장 정확도(%), 학습한 모델). use_aug 만 다르고 나머지는 모두 같음."""
    torch.manual_seed(seed)                           # 시작값 — 두 학습이 같은 출발점
    m = SmallCNN(len(CLASSES))
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(x_tr), generator=torch.Generator().manual_seed(seed * 100 + ep))  # 섞는 순서도 같게
        for k in range(0, len(x_tr), bs):
            idx = perm[k:k + bs]
            xb, yb = x_tr[idx], y_tr[idx]
            if use_aug:                               # ← 바꾼 조건은 이 한 줄뿐
                xb = torch.stack([my_augment(im) for im in xb])
            loss = F.cross_entropy(m(to_input(xb)), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return (predict(m, x_te).argmax(1) == y_te).float().mean().item() * 100, m

- 바꾼 것 — `use_aug` (내 증강을 넣는가) 하나
- 그대로 둔 것 — 학습 사진 500장 · 모델 모양 · 시작값 · 섞는 순서 · 에폭 · 묶음 크기 · 학습률 · 시험 사진 1000장 · 세는 방법

In [ ]:
seed = 0                                              # ◆ 시작값 — 0 으로 한 번 → 1 로 바꿔 한 번 더

acc_no, m_no = train_small(use_aug=False, seed=seed)
acc_aug, m_aug = train_small(use_aug=True, seed=seed)
print(f"시작값 {seed} — 증강 없음 {acc_no:.1f}% · 내 증강 {acc_aug:.1f}% · 차이 {acc_aug - acc_no:+.1f}%p")

- 확인 질문 1 — 시작값 0 한 번의 차이만 보고 "내 증강이 좋다(나쁘다)"라고 말해도 되는가
- 확인 질문 2 — 시작값을 1 로 바꿨을 때 차이의 **방향**이 같은가 · 증강 없음끼리 시작값만 바꿔도 몇 %p 움직이는가
- 흔한 에러
  - `NameError: name 'my_augment' is not defined` — 문제 2 칸을 실행하지 않았음 · 문제 2 의 함수 칸(또는 막혔을 때 칸)을 먼저 실행
  - 학습 중 모양 에러 — `my_augment` 가 사진 한 장(3, 32, 32)을 받아 한 장을 돌려주는지 확인 · 묶음을 돌려주면 안 됨
  - 너무 오래 걸림 — 무거운 변환(회전)을 여러 개 넣으면 느려짐 · 1분이 넘으면 `epochs=8` 을 5 로 줄여도 됨 (두 학습 모두 같이 줄어듦)

**막혔을 때 이어갈 코드** — 좌우 반전 증강으로 시작값 0 한 번

In [ ]:
def my_augment(img):
    x = img.float()
    if torch.rand(1).item() < 0.5:
        x = x.flip(-1)
    return x.clamp(0, 255)
(a0, m_no), (a1, m_aug) = train_small(False, 0), train_small(True, 0)
print(f"시작값 0 — 증강 없음 {a0:.1f}% · 좌우 반전 {a1:.1f}% · 차이 {a1 - a0:+.1f}%p")

### 문제 4 ★★ 도전 — 같은 두 모델을 흔든 시험 사진으로 견주기

- 할 일 — 방금 학습한 두 모델(`m_no` · `m_aug`)을 다시 학습하지 않고, 시험 사진을 **흔든 판**으로 바꿔 잼
- 조건 하나 — 시험 사진에 준 변화 · 두 모델 · 시험 사진 1000장은 그대로
- 확인 질문 — 원래 시험 사진에서는 차이가 작았는데 흔든 시험에서는 커진다면, 내 증강은 무엇을 연습시킨 것인가

In [ ]:
def shaken(x):
    return x.float().flip(-1)                         # ◆ 시험 사진에 줄 변화 — 예: 좌우 반전 · 밝기 (x.float() * 0.6).clamp(0, 255)


for name, m in (("증강 없음", m_no), ("내 증강", m_aug)):
    a0 = (predict(m, x_te).argmax(1) == y_te).float().mean().item() * 100
    a1 = (predict(m, shaken(x_te)).argmax(1) == y_te).float().mean().item() * 100
    print(f"{name} — 원래 시험 {a0:.1f}% · 흔든 시험 {a1:.1f}% · 떨어진 폭 {a0 - a1:.1f}%p")

### 문제 4 ★★★ 심화 — 시작값 셋으로 방향 세기

- 할 일 — 시작값 0 · 1 · 2 에서 두 조건을 모두 돌리는 반복문을 몇 줄로 짜고, 증강 쪽이 높은 경우가 몇 번인지 셈 (약 2~3분)
- 힌트 — `for sd in (0, 1, 2):` 안에서 `train_small(False, sd)[0]` · `train_small(True, sd)[0]` · 차이가 0 보다 크면 1 을 더함

In [ ]:
run_three = False                                     # ◆ True 로 바꾸면 실행 (시간이 걸림)
if run_three:
    up = 0
    # ◆ (채움) 시작값 0 · 1 · 2 반복 → 두 조건 학습 → 차이 출력 → 증강 쪽이 높으면 up += 1
    print(f"증강 쪽이 높은 경우 {up}번 / 3번")

## 문제 5 — 세 줄로 적기

아래 칸을 두 번 눌러 고칠 것. 문제 4(작은 통제 비교)의 결과로 적음 · 문제 4 를 못 했으면 문제 1~3 중 하나로 적음.

- **바꾼 것과 그대로 둔 것** —
- **예상과 실제** —
- **아직 모르는 것과 다음 비교** —

**예시** — 먼저 적어 본 뒤에 볼 것 (숫자는 자기 출력에서 옮겨 적음)

- 바꾼 것과 그대로 둔 것 — 내 증강(좌우 반전 + 밝기)을 넣는가만 바꿈 · 학습 사진 500장 · 모델 · 시작값 · 에폭 · 시험 사진 1000장은 그대로
- 예상과 실제 — 증강이 확실히 오를 줄 알았는데 두 시작값에서 차이가 (자기 출력) 이고, 시작값만 바꾼 흔들림도 (자기 출력) 이었음
- 아직 모르는 것과 다음 비교 — 학습이 길어지면 차이가 커질까 · 두 조건 모두 에폭을 늘리고 시작값을 셋 이상으로 다시 잼